# 01 — Explore network traffic data

**Goal:** understand the training data before making cleaning or modeling decisions.
This notebook does not train a model or modify the raw dataset. Download the
headered CSV using `data/README.md` first. Run cells in order.

A **feature** describes an observation; the **target** is the outcome we will later
predict. Here `label` is the binary target (0 normal, 1 attack). We reserve the
provided test set for final evaluation and explore only training data.


In [ ]:
from pathlib import Path
import sys
import platform
from importlib.metadata import version

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Support Jupyter launched from either the repository root or notebooks/.
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if not (project_root / "src" / "data_loader.py").is_file():
    raise RuntimeError("Start Jupyter from the network-ml-lab repository root.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_loader import load_dataset
from src.utils import RANDOM_STATE

print("Python:", platform.python_version())
print({package: version(package) for package in ["pandas", "numpy", "matplotlib"]})
print("Random seed for later experiments:", RANDOM_STATE)
data = load_dataset("training", data_dir=project_root / "data" / "raw")
display(data.head())


## Shape, columns, and types

Rows are observations; columns include features and labels. Pandas infers types,
so inspect whether numeric-looking fields were read as text.


In [ ]:
print(f"Rows: {data.shape[0]:,}; columns: {data.shape[1]}")
print("Column names:", data.columns.tolist())
display(data.dtypes.rename("dtype").to_frame())
data.info()


## Missing values and duplicates

Missingness may affect later imputation. Text placeholders such as `-` are not
automatically missing in pandas. Repeated rows deserve investigation before
removal; similar traffic can legitimately occur more than once.


In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": data.isna().sum(),
    "missing_percent": data.isna().mean() * 100,
}).sort_values("missing_count", ascending=False)
display(missing_summary)
print("Exact duplicate rows (after first occurrence):", data.duplicated().sum())
traffic_columns = data.columns.difference(["id", "label", "attack_cat"])
if len(traffic_columns):
    print("Repeated traffic feature rows (excluding ID and labels):",
          data.duplicated(subset=traffic_columns.tolist()).sum())
text_columns = data.select_dtypes(include=["object", "string"]).columns
if len(text_columns):
    display(data[text_columns].eq("-").sum().rename("dash_placeholder_count").to_frame())


## Target distribution

Class imbalance means one class appears more often. A future classifier could
achieve misleading accuracy by mostly predicting the majority class.


In [ ]:
target_counts = data["label"].value_counts().reindex([0, 1], fill_value=0)
target_summary = pd.DataFrame({
    "class": ["Normal", "Attack"],
    "count": target_counts.to_numpy(),
    "percent": target_counts.to_numpy() / len(data) * 100,
}, index=pd.Index([0, 1], name="label"))
display(target_summary)
if "attack_cat" in data.columns:
    display(data["attack_cat"].value_counts(dropna=False).rename("count").to_frame())

ax = target_summary.set_index("class")["count"].plot.bar(color=["steelblue", "coral"])
ax.set(title="Training target distribution", xlabel="Class", ylabel="Records")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## Numeric statistics and distributions

Mean and standard deviation describe scale; medians and quartiles help when
features are skewed. IDs and labels are excluded from feature plots. Infinities
are counted explicitly because `isna()` does not count them.


In [ ]:
numeric_features = data.select_dtypes(include="number").drop(
    columns=["id", "label", "attack_cat"], errors="ignore"
)
display(data.describe(include="number").T)
if not numeric_features.empty:
    display(pd.Series(np.isinf(numeric_features.to_numpy()).sum(axis=0),
                      index=numeric_features.columns, name="infinite_count").to_frame())

# Prefer familiar traffic quantities; fall back to available numeric features.
plot_columns = [name for name in ["dur", "sbytes", "dbytes", "spkts"]
                if name in numeric_features.columns]
if not plot_columns:
    plot_columns = numeric_features.columns[:4].tolist()
if plot_columns:
    # Replace infinity only in this plot copy; raw data remains unchanged.
    plot_data = numeric_features[plot_columns].replace([np.inf, -np.inf], np.nan)
    plot_data.hist(bins=40, figsize=(10, 6))
    plt.suptitle("Training numeric feature distributions (raw scale)")
    plt.tight_layout()
    plt.show()
else:
    print("No numeric traffic features available to plot.")


## Categorical feature example

Protocol counts give context for which traffic is represented. The top ten view
is a display choice; it does not remove categories from the dataset.


In [ ]:
if "proto" in data.columns:
    ax = data["proto"].value_counts(dropna=False).head(10).plot.bar(figsize=(8, 4))
    ax.set(title="Most frequent training protocols (top 10)",
           xlabel="Protocol", ylabel="Records")
    plt.tight_layout()
    plt.show()
else:
    print("No proto column in this CSV; skip the protocol plot.")


## Write your observations before moving on

Update `docs/findings.md`: which features are skewed, which values need
investigation, and how balanced are the labels? Do not infer predictive quality
from these plots.

**Leakage checkpoint:** `attack_cat` reveals attack status and `id` is an
identifier. Neither belongs in future binary model inputs. Fit imputation,
encoding, scaling, and feature selection only after separating training and
validation data, and only on the training subset. Keep the official test partition
reserved. Phase 2 will make those decisions; stop here for now.

Clear notebook outputs before committing. No figures or metrics are saved by
this notebook; the results folders are reserved for later experiments.
